# 27_anomaly_detection.ipynb

**13주차 · 2교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`13week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 7셀. 위에서부터 순서대로 실행합니다.

## 3-2. 정상만으로 학습

**셀 1** — 정상 = 신발류, 이상 = 가방

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

NORMAL, ANOMALY = [5, 7, 9], [8]        # 샌들·스니커즈·앵클부츠 / 가방  ★ 지정값
tf = transforms.ToTensor()
tr = datasets.FashionMNIST("data", train=True,  transform=tf)
te = datasets.FashionMNIST("data", train=False, transform=tf)

idx_tr = [i for i, (_, y) in enumerate(tr) if y in NORMAL]
train_dl = DataLoader(Subset(tr, idx_tr), batch_size=256, shuffle=True)
print("정상 학습 데이터 :", len(idx_tr), "장  (이상은 한 장도 안 쓴다 ★)")

**셀 2** — 1교시 모델 재사용해 학습

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, latent=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(),
                                     nn.Linear(256,128), nn.ReLU(), nn.Linear(128,latent))
        self.decoder = nn.Sequential(nn.Linear(latent,128), nn.ReLU(),
                                     nn.Linear(128,256), nn.ReLU(), nn.Linear(256,784),
                                     nn.Sigmoid(), nn.Unflatten(1, (1,28,28)))
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z

model = AutoEncoder(32).to(DEV)
opt, crit = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()
for ep in range(10):
    tot = 0.
    for xb, _ in train_dl:
        xb = xb.to(DEV); loss = crit(model(xb)[0], xb)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()*len(xb)
    print(f"epoch {ep+1:2d} | MSE {tot/len(idx_tr):.5f}")

## 3-3. 복원 오차 분포

**셀 3** — 샘플마다 복원 오차를 잰다 ★

In [ ]:
def recon_errors(ds, classes):
    idx = [i for i, (_, y) in enumerate(ds) if y in classes]
    dl  = DataLoader(Subset(ds, idx), batch_size=512)
    errs = []
    model.eval()
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(DEV)
            e = ((model(xb)[0] - xb) ** 2).mean(dim=[1, 2, 3])   # ★ 샘플별 MSE
            errs.append(e.cpu())
    return torch.cat(errs).numpy()

e_norm = recon_errors(te, NORMAL)
e_anom = recon_errors(te, ANOMALY)
print(f"정상 : 평균 {e_norm.mean():.5f} | 중앙값 {np.median(e_norm):.5f}")
print(f"이상 : 평균 {e_anom.mean():.5f} | 중앙값 {np.median(e_anom):.5f}  ← 크다 ★")

**셀 4** — 두 분포를 겹쳐 그린다 ★★

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.hist(e_norm, bins=60, alpha=0.6, label="정상 (신발류)", density=True)
plt.hist(e_anom, bins=60, alpha=0.6, label="이상 (가방)",   density=True)
thr = np.percentile(e_norm, 95)                       # ★ 정상의 95 백분위수
plt.axvline(thr, color="red", ls="--", label=f"임계값 (정상 95%) = {thr:.4f}")
plt.xlabel("복원 오차 (MSE)"); plt.ylabel("밀도")
plt.title("복원 오차 분포 — 정상 vs 이상")
plt.legend(); plt.tight_layout(); plt.show()

**셀 5** — 혼동행렬 (6주차 지표 ★)

In [ ]:
tp = (e_anom >  thr).sum(); fn = (e_anom <= thr).sum()
fp = (e_norm >  thr).sum(); tn = (e_norm <= thr).sum()
prec = tp / (tp + fp); rec = tp / (tp + fn)
print("            예측:정상  예측:이상")
print(f"실제:정상   {tn:7d}  {fp:9d}")
print(f"실제:이상   {fn:7d}  {tp:9d}")
print(f"\n정밀도 {prec:.3f} | 재현율 {rec:.3f} | F1 {2*prec*rec/(prec+rec):.3f}")

**셀 6** — 가장 오차가 큰 것들을 눈으로

In [ ]:
idx_a = [i for i, (_, y) in enumerate(te) if y in ANOMALY]
order = np.argsort(-e_anom)[:6]
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
model.eval()
for k, o in enumerate(order):
    x0 = te[idx_a[o]][0].unsqueeze(0).to(DEV)
    with torch.no_grad(): xh = model(x0)[0].cpu()
    axes[0, k].imshow(x0.cpu()[0, 0], cmap="gray"); axes[0, k].axis("off")
    axes[1, k].imshow(xh[0, 0],       cmap="gray"); axes[1, k].axis("off")
    axes[0, k].set_title(f"{e_anom[o]:.4f}", fontsize=9)
plt.suptitle("이상 입력(위)과 그 복원(아래) — 신발처럼 복원하려 애쓴다 ★")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★★**: 가방을 넣으면 모델이 **신발 비슷한 무언가로 복원**하려 합니다. **신발밖에 모르기 때문**입니다. 그래서 오차가 큽니다. 이 그림 한 장이 이상 탐지의 원리를 전부 설명합니다.

## 4-1. 트레이드오프

**셀 7** — 임계값을 바꿔 보며 표로 ★

In [ ]:
print("백분위  임계값     정밀도   재현율   F1")
for p in [80, 90, 95, 99]:
    t = np.percentile(e_norm, p)
    tp_ = (e_anom > t).sum(); fp_ = (e_norm > t).sum(); fn_ = (e_anom <= t).sum()
    pr = tp_/(tp_+fp_+1e-9); rc = tp_/(tp_+fn_+1e-9)
    print(f"  {p:3d}   {t:.5f}   {pr:.3f}   {rc:.3f}   {2*pr*rc/(pr+rc+1e-9):.3f}")